# Module 10 — The Memory Model, Scope, and Closures

**Part II: Core Skills** · Level: Intermediate · Prerequisites: Module 8

## What you'll learn

- Mutable vs immutable objects, identity, and aliasing
- Shallow and deep copies
- Scope rules: local, enclosing, global, built-in
- `global` and `nonlocal`
- Closures and the classic traps (mutable defaults, late binding)

---

## 10.1 Names refer to objects

In Module 2 you learned that a variable is a name bound to an object. Assignment never copies the object; it just adds another name.

In [ ]:
a = [1, 2, 3]
b = a                 # b is another name for the SAME list
b.append(4)
print(a)              # [1, 2, 3, 4]
print(a is b)         # True
print(id(a) == id(b)) # True

`id()` gives an object's identity; `is` compares identities; `==` compares values.

## 10.2 Mutable vs immutable

- **Immutable:** `int`, `float`, `str`, `bool`, `tuple`, `frozenset`, `None`. Operations create *new* objects.
- **Mutable:** `list`, `dict`, `set`, and most custom objects. They can change in place, so every name referring to them sees the change.

In [ ]:
a = [1, 2, 3]
b = "abc"
a.append(4)
print(a)
# b[0] = "z"      # TypeError: strings are immutable

s = "hello"
t = s
s += " world"          # creates a NEW string; t is unaffected
print(s, "|", t)

point = (0, 0)
# point[0] = 1     # TypeError

Passing a mutable object to a function lets the function modify the caller's data:

In [ ]:
def add_item(items, value):
    items.append(value)          # mutates the caller's list

shopping = ["milk"]
add_item(shopping, "eggs")
print(shopping)                  # ['milk', 'eggs']

def add_item_safe(items, value):
    return items + [value]       # returns a new list instead

print(add_item_safe(shopping, "tea"), shopping)

## 10.3 Shallow vs deep copy

Ways to copy a list: `list(x)`, `x[:]`, `x.copy()`, or `copy.copy(x)`. These are **shallow**: the outer container is new, but nested objects are still shared.

In [ ]:
import copy

original = [[1, 2], [3, 4]]
shallow = copy.copy(original)
shallow[0].append(99)
print(original)                  # [[1, 2, 99], [3, 4]]  <- changed!

original = [[1, 2], [3, 4]]
deep = copy.deepcopy(original)   # recursively copies nested objects
deep[0].append(99)
print(original, deep)            # original untouched

Another classic trap — repeating a mutable item:

In [ ]:
grid_bad = [[0] * 3] * 3         # three references to ONE row
grid_bad[0][0] = 1
print(grid_bad)                  # [[1, 0, 0], [1, 0, 0], [1, 0, 0]]

grid_ok = [[0] * 3 for _ in range(3)]    # a fresh row each time
grid_ok[0][0] = 1
print(grid_ok)

## 10.4 Mutable default arguments

Default values are evaluated **once**, when `def` runs. A mutable default is therefore shared between calls.

In [ ]:
def append_item(item, values=[]):      # BUG
    values.append(item)
    return values

print(append_item("a"))                # ['a']
print(append_item("b"))                # ['a', 'b']  <- surprise

def append_item_fixed(item, values=None):
    if values is None:
        values = []
    values.append(item)
    return values

print(append_item_fixed("a"))          # ['a']
print(append_item_fixed("b"))          # ['b']

## 10.5 Scope and the LEGB rule

Python resolves a name by searching, in order:

1. **L**ocal — inside the current function
2. **E**nclosing — in any outer function
3. **G**lobal — at module level
4. **B**uilt-in — names like `len` and `print`

In [ ]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(x)         # local
    inner()
    print(x)             # enclosing

outer()
print(x)                 # global

Variables assigned inside a function are **local** unless declared otherwise. A local variable with the same name as an outer one **shadows** it:

In [ ]:
count = 5

def demo():
    count = 10           # a new local; the global is untouched
    print(count)

demo()
print(count)             # 5

Never shadow built-ins such as `list`, `sum`, or `max`; it breaks later code in the same scope.

## 10.6 Namespaces

A **namespace** is a mapping from names to objects. Every module, function call, and class has its own, which is why two modules can both define `config` without colliding.

In [ ]:
import math

print(math.__name__)
print("sqrt" in dir(math))
print(list(globals())[:3])          # the module-level namespace
def f():
    local_one = 1
    return list(locals())
print(f())                          # ['local_one']

## 10.7 `global` and `nonlocal`

To **rebind** (not just read) a name from an outer scope, declare it.

In [ ]:
counter = 0

def increment():
    global counter       # refer to the module-level name
    counter += 1

increment(); increment()
print(counter)           # 2

def make_counter():
    count = 0
    def inner():
        nonlocal count   # refer to the enclosing function's variable
        count += 1
        return count
    return inner

next_id = make_counter()
print(next_id(), next_id())    # 1 2

Reading a global needs no declaration, and mutating a global *object* (e.g. appending to a global list) does not either. Prefer passing values in and returning results out; `global` makes code harder to test.

## 10.8 Closures and nested functions

A **closure** is a function that remembers variables from the scope where it was defined, even after that scope has finished.

In [ ]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor      # `factor` is captured
    return multiply

triple = make_multiplier(3)
double = make_multiplier(2)
print(triple(4), double(4))    # 12 8

def outer():
    def inner():
        return "inside"
    return inner

print(outer()())               # inside

print([cell.cell_contents for cell in triple.__closure__])   # [3]

Closures give you lightweight configurable functions and are the foundation of decorators (Module 17).

### The late-binding trap

A closure captures the *variable*, not its value at the time of creation.

In [ ]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])               # [2, 2, 2]  <- all see the final i

funcs = [lambda i=i: i for i in range(3)]  # bind the current value as a default
print([f() for f in funcs])               # [0, 1, 2]

## Common pitfalls

- Believing `b = a` copies a list.
- Using a shallow copy on nested data.
- Mutable default arguments.
- `UnboundLocalError`: assigning to a name inside a function makes it local for the *whole* function, so reading it earlier fails.
- Overusing `global`.

## Practice

1. Predict, then verify, what `x` is after `y = x = []; y.append(1)`.
2. Write `make_accumulator(start)` returning a function that adds its argument to a running total.
3. Fix `def add_tag(tag, tags=[]): ...` so each call starts with a fresh list.